# S4Casting: Training Pipeline

This Source Code Form is subject to the terms of the Mozilla Public  License, v. 2.0. If a copy of the MPL was not distributed with this file, You can obtain one at http://mozilla.org/MPL/2.0/.
This notebook demonstrates the **training pipeline**, as in *scripts/train.py*.
It walks through loading the configuration, setting up the model, optimizer, trainer, and running the training loop.

The pipeline includes:
1. **Configuration loading** — reading system and model setup from `cuda.toml`.
2. **Machine and model initialization** — creating compute, model, optimizer, and trainer components.
3. **Training** — running the full model training loop.

## Prerequisites

Before running this notebook:
- ✅ Complete `00_data_preparation.ipynb` (generates training data)


## Load Configuration File

The project uses a `.toml` configuration file (e.g., `example_config/cuda.toml`) to define:

- **Hardware settings** (e.g., GPU, distributed training)
- **Model parameters**
- **Optimizer and scheduler settings**
- **Training and evaluation setup**
- **I/O paths for datasets and checkpoints**

The configuration is parsed and validated through the `s4config.Configuration` class.

We defined a smaller version of the model as this is a demonstration notebook and we want to keep training time short.

In [ ]:
import pathlib

import tomlkit
import torch

from s4casting.core.config import Configuration

with pathlib.Path("../configs/cuda-tiny.toml").open("r", encoding="utf-8") as f:
    config_data = tomlkit.load(f)

# Use smaller dataset for faster training in the notebook, set before config instantiation and pydantic validation
config_data["io"]["features"]["measurements"]["location"] = "data/notebook_test_data/external_data_wrapped.json"
config_data["io"]["features"]["measurements"]["loader"] = "parquet"
config = Configuration(**config_data.unwrap())

# Use GPU if available
config.machine.device_kind = "cuda" if torch.cuda.is_available() else "cpu"

# Reduce number of training steps for faster training in the notebook
config.training.maximum_steps = 50
# Increase learning rate for faster convergence in the notebook
config.optimizer.learning_rate = 1e-2
# Reduce model size for faster training in the notebook
config.model.ssm.n_layers = 3
config.model.latent_dim = 64
config.model.context_window = [11]
config.model.predict_width = 2

# Use Gaussian Mixture Model output head with Negative Log Likelihood loss
config.model.loss.loss = "nll"
config.model.output_head.arch = "gmm"

# Reduce number of Gaussians for faster training in the notebook
config.model.output_head.n_gaussians = 2

config

# Training Pipeline Overview

This notebook demonstrates the training pipeline for the S4Casting project on the sinusoid dataset created in the previous notebook. The pipeline is designed to handle time-series forecasting tasks using advanced deep learning models. It includes components for data loading, preprocessing, model training, evaluation, and benchmarking.

Key components of the pipeline:
1. **Machine Configuration**: Sets up the hardware environment (CPU/GPU, distributed training).
2. **Model Container**: Encapsulates the model, optimizer, and training configuration.
3. **Optimizer**: Manages the optimization algorithm and learning rate scheduling.
4. **Scheduler**: Adjusts the learning rate during training based on a predefined schedule.
5. **Trainer**: Handles the training loop, including loss computation and backpropagation.
6. **Checkpointer**: Saves and loads model checkpoints during training.
7. **Evaluator**: Evaluates the model's performance on validation data.
8. **Batcher**: Manages data loading and preprocessing for training, validation, and benchmarking.
9. **Benchmarker**: Benchmarks the model's performance on specific tasks or datasets.
10. **Context Manager**: Manages the overall training context.

> **Note:** The next cell may appear to pause during initial setup. The first run can take 5–10 minutes while extensions compile. To monitor progress, run in a terminal: `watch -n 1 "ls -lh ~/.cache/torch_extensions/py312_cu128/selective_scan_cuda/"`

In [ ]:
from s4casting import factories as fc
from s4casting.core.context import Context

machine = fc.provide_machine(config.machine, config.run.seed)
model_container = fc.provide_model_container(config.model, config.io, machine)
optimizer = fc.provide_optimizer(config.optimizer, model_container.raw_model.parameters())
scheduler = fc.provide_scheduler(config.scheduler, optimizer)
trainer = fc.provide_trainer(config=config.training, optimizer=config.optimizer, machine=machine)
checkpointer = fc.provide_checkpointer(config.io, trainer.hooks)
evaluator_head = fc.provide_evaluator_head(config.model, trainer.hooks)
evaluator = fc.provide_evaluation(trainer.hooks, evaluator_head)
benchmarker = fc.provide_benchmark(config.benchmarking, trainer.hooks, evaluator_head)
batcher = fc.provide_batcher(
    config.io,
    config.model,
    config.training,
    config.validation,
    config.benchmarking,
    config.run,
    machine,
    trainer.hooks,
)

context = Context(
    configuration=config,
    model_container=model_container,
    optimizer=optimizer,
    scheduler=scheduler,
    machine=machine,
    trainer=trainer,
    checkpointer=checkpointer,
    evaluator=evaluator,
    benchmarker=benchmarker,
    batcher=batcher,
)

## Model Inspection

Before starting training, we inspect the initialized model and its parameters.

This `S4Model` combines:
- **S6/Mamba sequence layers** — for efficient temporal modeling.  
- **Gaussian Mixture Model (GMM) output head** — for probabilistic forecasting rather than point predictions.

The summary below shows the model’s total and trainable parameters, memory size, and configuration details.

In [ ]:
import torch

total = sum(p.numel() for p in model_container.model.parameters())
trainable = sum(p.numel() for p in model_container.model.parameters() if p.requires_grad)
dtype = getattr(config.model, "internal_dtype", torch.float32)
print("=== Run Summary ===")
print(f"Model: {type(model_container.model).__name__}")
print(f"Parameters: total={total:,}, trainable={trainable:,} (~{(trainable * 4 / (1024**2)):.1f} MB)")
print(f"Max steps: {config.training.maximum_steps}")
print(f"Eval every: {config.training.evaluation_interval} steps")
print(f"Checkpoint every: {config.training.checkpoint_interval} steps")
print(f"The model object: {model_container.model}")

## Dataset and Mask Visualization

Each training sample consists of:
- `X`: input context (past time steps)
- `Y`: True value of the target forecast window
- `Xm`, `Ym`: corresponding binary masks (1 = active, 0 = masked)

The chart below shows the split between **context** and **prediction** windows in one example.

In [ ]:
from torch.utils.data import DataLoader

from s4casting.data.utils import collate_single_interval

N = len(batcher.train)
print(f"The training dataset has {N} samples")
first, _ = next(
    iter(
        DataLoader(
            batcher.train,
            batch_size=1,
            num_workers=0,
            shuffle=True,
            pin_memory=False,
            persistent_workers=False,
            collate_fn=collate_single_interval,
        )
    )
)
X, Xm, Y, Ym = (t for t in first)

total_minutes = N * X.shape[1] * config.model.base_sample_interval_minutes
total_days = total_minutes / (60 * 24)
total_years = total_days / 365.25

print(f"Per sample: {X.shape[1]} steps")
print(f"Dataset samples: {N}")
print(f"Approx coverage: {total_minutes} minutes, {total_days:.2f} days ≈ {total_years:.2f} years")

print(f"Input context shape: {X.shape}")
print(f"Input mask shape: {Xm.shape}")
print(f"Output shape: {Y.shape}")
print(f"Output mask shape: {Ym.shape}")

In [ ]:
import matplotlib.pyplot as plt

# Count where masks are 1
input_length = (Xm[0] == 1).sum().item()
pred_length = (Ym[0] == 1).sum().item()

fig, ax = plt.subplots(figsize=(10, 2))
ax.barh(0, input_length, label=f"Context/Input: {input_length} steps")
ax.barh(0, pred_length, left=input_length, label=f"Prediction/target: {pred_length} steps")
ax.set_xlim(0, X.shape[1])
ax.set_yticks([])
ax.legend()
ax.set_xlabel("Time Steps")
ax.set_title("Context vs Prediction Window Split");

## Example Training Sample

This plot visualizes a single training sample across multiple days.  
The left section represents the **context (input)**, and the right section shows the **forecast (target)**.


In [ ]:
import numpy as np

predict_width_samples = (config.model.predict_width * 24 * 60) // config.model.base_sample_interval_minutes
input_width_samples = (
    (config.model.context_window[0] * 24 * 60) // config.model.base_sample_interval_minutes
) - predict_width_samples
days_per_sample = config.model.context_window[0]
samples_per_day = 96  # 24 hours x 4 samples/hour

plt.figure(figsize=(20, 6))
plt.plot(np.arange(input_width_samples), X[0, :-192].squeeze(-1), label="Example context")
plt.plot(
    np.arange(input_width_samples, input_width_samples + predict_width_samples),
    Y[0, -192:].squeeze(-1),
    label="Example target window",
)

for day in range(0, days_per_sample):
    x_pos = day * samples_per_day
    plt.axvline(x=x_pos, color="gray", linestyle="--", alpha=0.3)
    plt.text(x_pos + samples_per_day / 2, plt.ylim()[0], f"Day {day + 1}", ha="center", va="bottom")
plt.title("Training data sample")
plt.xlabel("Time")
plt.ylabel("Load")
plt.legend()
plt.show()

## Training Loop

The following loop performs model training for the specified number of steps.  
It periodically visualizes the loss curve, showing how the model learns over time.

At each step:
1. A batch of data is loaded to the GPU.  
2. The model predicts Gaussian mixture parameters.  
3. Loss is computed and backpropagated.  
4. Gradients are clipped (if enabled) and optimizer updates weights.

The plotted loss provides an indication of convergence.

In [ ]:
from IPython.display import clear_output
from IPython.display import display as ipy_display
from tqdm import tqdm

from s4casting.core.functional import select_rate

total_loss = 0
losses = []
steps = []

context.optimizer.zero_grad(set_to_none=True)
plot_display_freq = 5
fig, ax = plt.subplots(figsize=(10, 5))

# basic training loop, do with tqdm

for iteration, step in enumerate(tqdm(range(config.training.maximum_steps), desc="Training")):
    # loading data to device (cpu/gpu)
    (X, Xm, Y, Ym), batch_cfg = next(
        iter(
            DataLoader(
                batcher.train,
                batch_size=config.training.batch_size,
                num_workers=0,
                shuffle=True,
                pin_memory=False,
                persistent_workers=False,
                collate_fn=collate_single_interval,
            )
        )
    )
    X = X.to(context.machine.torch_device).float()
    Y = Y.to(context.machine.torch_device).float()
    Xm = Xm.to(context.machine.torch_device).float()
    Ym = Ym.to(context.machine.torch_device).float()

    input_interval = batch_cfg.sample_interval_minutes.to(context.machine.torch_device)
    output_interval = select_rate(input_interval, config.model.output_sample_intervals_minutes)

    # input data to model
    _, loss = context.model_container.model(X, Xm, input_interval, output_interval, Y, Ym)
    loss.backward()
    total_loss += loss.item()

    # track model loss
    losses.append(loss.item())
    steps.append(iteration)

    if step != 0 and step % plot_display_freq == 0:
        clear_output(wait=True)
        ax.clear()
        ax.plot(steps, losses, "k", linewidth=2, alpha=0.6, marker="o")
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title(f"Training loss - step {iteration}: {loss.item():.3f}")
        ipy_display(fig)

    if trainer._optimizer.gradient_clipping:
        torch.nn.utils.clip_grad_norm_(context.model_container.model.parameters(), trainer._optimizer.gradient_clipping)

    context.optimizer.step()
    context.optimizer.zero_grad(set_to_none=True)
    if step == 0:
        checkpointer.save(context, iteration=1)

checkpointer.save(context, iteration=None)

## Summary & Next Steps

You've successfully trained a basic S4Casting model!

### Next Steps
1. Try to run inference using the trained model in `02_inference.ipynb`.
2. Experiment with different model architectures and hyperparameters.
3. Try training on your own datasets formatted with `DatasetFormatter`.
